### Markdown header

In [1]:
# Imports - PYTHON PHASE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML

# Plotly imports
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

### Loading data
Need to load data for future operations

In [2]:
# Load indices data - EXPANDED DATA LOADING
data_path = Path("../equities/indicies.parquet")

# Check if file exists
if not data_path.exists():
    print(f"⚠️ File not found at {data_path.resolve()}")
    print(f"Current working directory: {Path.cwd()}")
    print("Available files in equities/:")
    equities_path = Path("../equities")
    if equities_path.exists():
        for f in equities_path.iterdir():
            print(f"  - {f.name}")
else:
    # Load the data
    df_raw = pd.read_parquet(data_path)
    
    print(f"✅ Data loaded successfully")
    print(f"Shape: {df_raw.shape}")
    print(f"Columns: {df_raw.columns.tolist()}")
    print(f"Data types:\n{df_raw.dtypes}")
    
    # Check if there's a datetime column that needs to be set as index
    datetime_cols = df_raw.select_dtypes(include=['datetime64']).columns
    if len(datetime_cols) > 0:
        print(f"\n✅ Found datetime column: {datetime_cols[0]}")
        df_raw = df_raw.set_index(datetime_cols[0])
        print(f"Index set to: {df_raw.index.name}")
    
    # Display first few rows
    print(f"\n📊 First few rows:")
    display(df_raw.head(10))

⚠️ File not found at C:\Personal\Business & Investments\Trading portfolio\Cogilator\equities\indicies.parquet
Current working directory: c:\Personal\Business & Investments\Trading portfolio\Cogilator\btest
Available files in equities/:


In [3]:
# Calculate returns and momentum grouped by ticker
print("="*60)
print("RETURNS & MOMENTUM CALCULATION (ALL TICKERS)")
print("="*60)

# Sort by ticker and date
df_raw_sorted = df_raw.sort_index()

# Calculate returns per ticker
def calculate_returns_momentum(ticker_data):
    """Calculate returns and momentum for a single ticker"""
    ticker_data = ticker_data.sort_index()
    
    # Daily returns
    daily_ret = ticker_data['close'].pct_change()
    
    # 6-month momentum (126 trading days)
    momentum_6m = ticker_data['close'].pct_change(periods=126)
    
    return pd.DataFrame({
        'close': ticker_data['close'],
        'daily_ret': daily_ret,
        'momentum_6m': momentum_6m
    })

# Apply function to each ticker
returns_by_ticker = {}
for ticker in sorted(df_raw['ticker'].unique()):
    ticker_data = df_raw[df_raw['ticker'] == ticker]
    returns_by_ticker[ticker] = calculate_returns_momentum(ticker_data)

print(f"\n📊 Returns calculated for {len(returns_by_ticker)} tickers:")
for ticker, ret_df in returns_by_ticker.items():
    print(f"  {ticker}: {len(ret_df)} records, returns mean={ret_df['daily_ret'].mean()*100:.4f}%")

# Store for later use
print("\n✅ Returns and momentum calculated for all tickers")

RETURNS & MOMENTUM CALCULATION (ALL TICKERS)


NameError: name 'df_raw' is not defined

In [ ]:
# Ticker selector and analysis with synchronized charts (Plotly native dropdown)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("="*60)
print("TICKER SELECTION & ANALYSIS")
print("="*60)

available_tickers = sorted(df_raw['ticker'].unique())
print(f"\n📊 Available tickers: {available_tickers}")

# Color scheme - matches codebase dark theme
THEME_COLORS = {
    'bg': '#0b1220',
    'panel': '#0f1b33',
    'border': '#1f2d4d',
    'text': '#e6edf7',
    'muted': '#a9b7d0',
    'grid': 'rgba(31, 45, 77, 0.3)',
    'spike': 'rgba(119, 184, 255, 0.5)',
    'price': '#1f77b4',
    'positive': '#2ca02c',
    'negative': '#d62728',
    'momentum': '#9467bd',
}

# Create figure with subplots
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Price", "Daily Returns (%)", "6-Month Momentum"),
    specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
    vertical_spacing=0.12,
    row_heights=[0.35, 0.35, 0.3]
)

# Add traces for each ticker (initially hidden except first)
for i, ticker in enumerate(available_tickers):
    ticker_ret = returns_by_ticker[ticker].copy()
    ticker_ret['momentum_6m'] = ticker_ret['close'].pct_change(periods=126)
    daily_returns = ticker_ret['daily_ret'].copy()
    momentum_values = ticker_ret['momentum_6m'].copy()
    
    visible = (i == 0)
    
    # Price trace
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=ticker_ret['close'],
            name=f'{ticker} Price', mode='lines',
            line=dict(color=THEME_COLORS['price'], width=2),
            fill='tozeroy', fillcolor='rgba(31, 119, 180, 0.1)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>',
            visible=visible
        ), row=1, col=1
    )
    
    # Returns trace
    colors = [THEME_COLORS['positive'] if x > 0 else THEME_COLORS['negative'] for x in daily_returns]
    fig.add_trace(
        go.Bar(
            x=ticker_ret.index, y=daily_returns * 100,
            name=f'{ticker} Return', marker=dict(color=colors),
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Return: %{y:.3f}%<extra></extra>',
            visible=visible
        ), row=2, col=1
    )
    
    # Momentum trace
    fig.add_trace(
        go.Scatter(
            x=ticker_ret.index, y=momentum_values,
            name=f'{ticker} Momentum', mode='lines',
            line=dict(color=THEME_COLORS['momentum'], width=2),
            fill='tozeroy', fillcolor='rgba(148, 103, 189, 0.2)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Momentum: %{y:.4f}<extra></extra>',
            visible=visible
        ), row=3, col=1
    )

# Create dropdown buttons
buttons = []
n_traces_per_ticker = 3
for i, ticker in enumerate(available_tickers):
    visibility = [False] * (len(available_tickers) * n_traces_per_ticker)
    for j in range(n_traces_per_ticker):
        visibility[i * n_traces_per_ticker + j] = True
    
    buttons.append(dict(
        label=ticker,
        method='update',
        args=[{'visible': visibility},
              {'title': f'<b>Returns & Momentum Analysis - {ticker}</b>'}]
    ))

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.0,
        xanchor='left',
        y=1.15,
        yanchor='top',
        bgcolor=THEME_COLORS['panel'],
        bordercolor=THEME_COLORS['border'],
        font=dict(color=THEME_COLORS['text'])
    )],
    height=900,
    title_text=f"<b>Returns & Momentum Analysis - {available_tickers[0]}</b>",
    hovermode='x unified',
    plot_bgcolor=THEME_COLORS['panel'],
    paper_bgcolor=THEME_COLORS['bg'],
    font=dict(family="Arial, sans-serif", size=11, color=THEME_COLORS['text']),
    margin=dict(l=70, r=30, t=100, b=60),
    showlegend=False,
    title_font=dict(size=14, color=THEME_COLORS['text'])
)

fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(title_text="Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Return (%)", row=2, col=1)
fig.update_yaxes(title_text="Momentum", row=3, col=1)

fig.update_xaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikemode='across', spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)
fig.update_yaxes(
    gridcolor=THEME_COLORS['grid'],
    tickfont=dict(color=THEME_COLORS['muted']),
    showspikes=True, spikethickness=1.5,
    spikecolor=THEME_COLORS['spike']
)

print("\n✅ Use the dropdown menu (top-left) to select a ticker")
fig.show()

TICKER SELECTION & ANALYSIS

📊 Available tickers: ['CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX']

✅ Select a ticker from the dropdown below:


Dropdown(description='Select Ticker:', options=('CAC', 'CCMP', 'DAX', 'IBEX', 'MIB', 'SPX', 'UKX'), style=Desc…

Output()

---

## Strategy 2: External Alpha Capture

### Overview
This strategy leverages **analyst estimates** provided by **FactSet** to capture alpha from external information signals. Unlike pure price-based momentum strategies, External Alpha Capture incorporates fundamental analyst expectations to identify mispricings before they are fully reflected in market prices.

### Data Source
- **Provider**: FactSet
- **Data Type**: Analyst Estimates
- **Key Fields**:
  - EPS estimates (quarterly/annual)
  - Revenue estimates
  - Estimate revisions
  - Consensus vs. individual analyst forecasts
  - Surprise metrics (actual vs. expected)

### Signal Rationale
Analyst estimates contain forward-looking information that can predict future price movements:
1. **Estimate Revisions**: Upward/downward revisions often precede price adjustments
2. **Earnings Surprises**: Stocks beating expectations tend to outperform
3. **Dispersion**: High analyst disagreement may indicate uncertainty or opportunity
4. **Momentum in Estimates**: Trending improvements in forecasts signal positive fundamentals

### Signals to Explore
*(Awaiting user input on specific signals to focus on)*

In [ ]:
# Strategy 2: External Alpha Capture - Data Setup
# ================================================
# Data Source: FactSet Analyst Estimates

print("="*60)
print("STRATEGY 2: EXTERNAL ALPHA CAPTURE")
print("="*60)
print("\n📊 Data Source: FactSet Analyst Estimates")
print("\n🔑 Key estimate fields to consider:")
print("   - EPS estimates (FY1, FY2)")
print("   - Revenue estimates")
print("   - Estimate revisions (1M, 3M changes)")
print("   - Earnings surprise history")
print("   - Analyst recommendation changes")

# TODO: Load FactSet analyst estimates data
# estimates_path = Path("../data/factset_estimates.parquet")

print("\n⏳ Awaiting signal specification...")